# Regression model results

This notebook programmatically uses the project's prediction services and visualizes model quality.

In [ ]:
from collections.abc import Iterator

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from cities import City
from museums import Museum
from persistence.museum_repository import MuseumRepository
from prediction.linear_regression_prediction_service import LinearRegressionPredictionService
from prediction.sgd_regressor_prediction_service import SgdRegressorPredictionService

In [ ]:
class InMemoryMuseumRepository(MuseumRepository):
    def __init__(self, museums: list[Museum]) -> None:
        self._museums = museums

    def find_all(self, batch_size: int = 1000) -> Iterator[Museum]:
        for museum in self._museums:
            yield museum

    def save_all(self, museums: list[Museum]) -> list[Museum]:
        return museums


def build_linear_museum_data() -> list[Museum]:
    populations = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
    museums: list[Museum] = []

    for index, population in enumerate(populations, start=1):
        city = City(id=index, name=f"City {index}", population=population, country="CA")
        museums.append(
            Museum(
                id=index,
                name=f"Museum {index}",
                annual_visitor=2 * population + 5,
                city=city,
            )
        )

    return museums

In [ ]:
museums = build_linear_museum_data()
X = np.array([museum.city.population for museum in museums])
y_true = np.array([museum.annual_visitor for museum in museums])

linear_service = LinearRegressionPredictionService(InMemoryMuseumRepository(museums))
sgd_service = SgdRegressorPredictionService(
    InMemoryMuseumRepository(museums),
    batch_size=2,
    epochs=400,
)

y_pred_linear = np.array([linear_service.predict(int(population)) for population in X])
y_pred_sgd = np.array([sgd_service.predict(int(population)) for population in X])

In [ ]:
def metrics(y_actual: np.ndarray, y_predicted: np.ndarray) -> dict[str, float]:
    return {
        "mae": float(mean_absolute_error(y_actual, y_predicted)),
        "rmse": float(np.sqrt(mean_squared_error(y_actual, y_predicted))),
        "r2": float(r2_score(y_actual, y_predicted)),
    }

linear_metrics = metrics(y_true, y_pred_linear)
sgd_metrics = metrics(y_true, y_pred_sgd)

linear_metrics, sgd_metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_true, y_pred_linear, label="LinearRegression", color="tab:blue")
axes[0].scatter(y_true, y_pred_sgd, label="SGDRegressor", color="tab:orange")
line_min = min(y_true.min(), y_pred_sgd.min(), y_pred_linear.min())
line_max = max(y_true.max(), y_pred_sgd.max(), y_pred_linear.max())
axes[0].plot([line_min, line_max], [line_min, line_max], "k--", label="Ideal")
axes[0].set_title("Predicted vs Actual Visitors")
axes[0].set_xlabel("Actual annual visitors")
axes[0].set_ylabel("Predicted annual visitors")
axes[0].legend()

linear_residuals = y_true - y_pred_linear
sgd_residuals = y_true - y_pred_sgd
axes[1].hist(linear_residuals, bins=8, alpha=0.6, label="LinearRegression", color="tab:blue")
axes[1].hist(sgd_residuals, bins=8, alpha=0.6, label="SGDRegressor", color="tab:orange")
axes[1].set_title("Residual Distribution (Actual - Predicted)")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

## Metric comparison

- MAE: mean absolute error
- RMSE: root mean squared error
- R2: coefficient of determination

In [ ]:
labels = ["MAE", "RMSE", "R2"]
linear_values = [linear_metrics["mae"], linear_metrics["rmse"], linear_metrics["r2"]]
sgd_values = [sgd_metrics["mae"], sgd_metrics["rmse"], sgd_metrics["r2"]]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width / 2, linear_values, width, label="LinearRegression", color="tab:blue")
ax.bar(x + width / 2, sgd_values, width, label="SGDRegressor", color="tab:orange")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title("Model Metrics")
ax.legend()

for index, value in enumerate(linear_values):
    ax.text(index - width / 2, value, f"{value:.3f}", ha="center", va="bottom")

for index, value in enumerate(sgd_values):
    ax.text(index + width / 2, value, f"{value:.3f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()